# E-Mail Classifier
Here we use Hugging face pretrained model. We do NOT define these classes in advance.


**You provide candidate labels at inference time.**

In [6]:
# check package version
import importlib.metadata

# List the distribution package names
packages = ["transformers", "torch"]

for package in packages:
    try:
        version = importlib.metadata.version(package)
        print(f"{package} version: {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package} is not installed in this environment.")

transformers version: 4.46.0
torch version: 2.14.0


In [2]:
# NOTE: First time it took
import time
import torch
from transformers import pipeline

# 1. Automatically detect and leverage your laptop's GPU
device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {'GPU (CUDA)' if device == 0 else 'CPU'}\n")

# 2. Initialize the Zero-Shot pipeline
print("Loading model (this takes a few seconds the first time)...")

# This is 1.63GB file
classifier = pipeline("zero-shot-classification",
                      model="facebook/bart-large-mnli", 
                      device=device
)

print("Model loaded successfully!\n" + "="*50)


Using device: CPU

Loading model (this takes a few seconds the first time)...


config.json: 0.00B [00:00, ?B/s]

C:\Users\ash32\Desktop\education\play_langchain_langgraph\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ash32\.cache\huggingface\hub\models--facebook--bart-large-mnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model loaded successfully!


In [3]:
def run_demo(email_text, categories):
    print(f"\n📧 Input Email:\n{email_text.strip()}")
    print(f"\n🏷️ Candidate Categories: {categories}")
    
    # Time the performance to show off GPU speed
    start_time = time.time()
    result = classifier(email_text, categories)
    end_time = time.time()
    
    print(f"⏱️ Inference Time: {end_time - start_time:.4f} seconds")
    print("\n📊 Classification Results:")
    
    # Print sorted results
    for label, score in zip(result['labels'], result['scores']):
        print(f"  - {label}: {score*100:.1f}%")
    print("="*50)


In [4]:

# --- LIVE DEMO SCENARIOS ---

# Scenario A: Standard Support Routing
email_1 = """
Hello, I am writing to request a full refund for my order #10293. 
The software keeps crashing every time I click the export button, and I cannot use it for work.
"""
categories_1 = ["Refund Request", "Technical Support", "Sales Inquiry", "Spam"]
run_demo(email_1, categories_1)



📧 Input Email:
Hello, I am writing to request a full refund for my order #10293. 
The software keeps crashing every time I click the export button, and I cannot use it for work.

🏷️ Candidate Categories: ['Refund Request', 'Technical Support', 'Sales Inquiry', 'Spam']
⏱️ Inference Time: 3.8610 seconds

📊 Classification Results:
  - Refund Request: 67.2%
  - Spam: 18.5%
  - Technical Support: 7.4%
  - Sales Inquiry: 6.9%


In [5]:
# Scenario B: Dynamically changing categories live in class
email_2 = """
Hey! Check out these amazing discounts on premium leather jackets! 
Click this link now to get 50% off before midnight!!!
"""
# You can show the class how changing this list changes the model's objective on the fly
categories_2 = ["Urgent Business", "Marketing/Spam", "Personal Chat"]
run_demo(email_2, categories_2)



📧 Input Email:
Hey! Check out these amazing discounts on premium leather jackets! 
Click this link now to get 50% off before midnight!!!

🏷️ Candidate Categories: ['Urgent Business', 'Marketing/Spam', 'Personal Chat']
⏱️ Inference Time: 2.6344 seconds

📊 Classification Results:
  - Marketing/Spam: 40.4%
  - Urgent Business: 31.5%
  - Personal Chat: 28.1%
